# Calculating Geodiveristy metrics for a model ensemble.


## Import Modules

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pyvista as pv

import geodiv_functions as gdf

## Load data

This code expects the input files to have the following dimensions: $(n model, x_{idx}, y_{idx}, z_{idx})$. Values are the property being analysed.


In [ ]:
# load data
path_to_data = '{path_to_your_data}'  # Replace with your actual path

ensemble_tensor = np.load(path_to_data)  # Load data

n_m, nx, ny, nz = ensemble_tensor.shape

# Flatten the tensor to a 2D array for PCA
# New shape will be (n_m, nx*ny*nz)
X = ensemble_tensor.reshape(n_m, -1)

## plot a model

Using pyvista, select a model and plot it in 3D

In [ ]:
model_to_display = 57

# Get dimension sizes from ensemble_list
ensemble_shape = ensemble_tensor.shape
print(f"Full ensemble_list shape: {ensemble_shape}")


# Create a list of dimension sizes
dimension_list = [n_m, nx, ny, nz]
print(f"Dimension list: {dimension_list}")

model_plot = ensemble_tensor[model_to_display]



In [ ]:
# Extract property values and reshape to 3D grid
values = model_plot.reshape((nx, ny, nz), order='F')

# Create a pyvista structured grid
# Create coordinate arrays for the grid
x = np.arange(nx)
y = np.arange(ny)
z = np.arange(nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Create structured grid
grid = pv.StructuredGrid(X, Y, -Z)
grid['density'] = values.flatten(order='F')

# Plot in 3D
plotter = pv.Plotter()
plotter.add_mesh(grid, scalars='density', cmap='viridis', opacity=0.7)
plotter.add_mesh(grid)
plotter.show()
plotter.add_title(f'Model {model_to_display} Density Plot')

## Calculate diversity metrics for each model

The metrics are statistical, geometrical and topological; 
### Statistical
* average density for each model
* variance
* skew
* minimum
* maximum

### Geometrical
* centre of mass: the location of the 
* orientation and stretch of a density mass

### Topological 

* Global Moran's I - spatial autocorrelation, how clumpy the density distribution is

```math
I = {N \over S_0} {\sum{_i} \sum{_j} w_{ij} (z_i)(z_j) \over \sum_i z^2_i}
```
Where:

* $N$ is the total number of voxels.
* $z_i = (\rho_i - \bar{\rho})$ is the deviation of density at voxel i for the mean density $\bar{\rho}$.
* $w{_ij}$ are the spatial weights indicating connectivity (= 1 if voxels are neighbours, = 0 otherwise).
* $S_0 = \sum{_i} \sum{_j} w_{ij} $ is the sum of all weights.

#### Interpretation of Moran's I
* $I \approx 1.0: $ Highly clustered into distinct bodies and blobs.
* $I \approx 0: $ Random. Property values distributed like white noise.
* $I \approx -1.0: $ Perfectly dispersed like a checkerboard pattern. High property values are always adjacent to low. 

### Possible additions
* volume of high or low density zones (above or below a threshold)
* semi-variogram parameters



## Execute functions
* extract metrics
* run the PCA

In [ ]:
morans = gdf.extract_morans_i(X, neighborhood_selection="all")

In [ ]:
# Usage:
features = gdf.extract_metrics_4d(ensemble_tensor)
final_df, scores, pca_obj = gdf.analyze_ensemble_pca(features)

## Visualise results

In [ ]:
gdf.plot_pca_results(pca_obj, scores[:,:2], features.columns) # scores[:,:2] are the coordinates of each model in the PC1-PC2 space, features.columns are the metric names for the loading vectors

## PCA and Hotelling’s T2 Calculation

Hotelling's $T^2$ score identifies outliers by measuring the multivariate distance from the centre of the ensemble. For this case, the multivariates are the geodiversity metrics.

The T2 statistic for a model realization i in the PCA space is calculated as:
```math
T^2​ = n(\bar{x} - \mu)'S^-1(\bar{x} - \mu)
```
Where
* $n$ is the number of samples (inversion models in this case)
* $S^-1$ is the inverse covariance matrix.
* $(\bar{x} - \mu)$ is the distance from the mean

The interpretation is that a model with a high $T^2$ score is unique when compared to the rest of the models as characterised by the geodiversity metrics. The following control charts plot all the models and their distance from the multivariate centre of the ensemble. Outliers are defined as exceeding a predefined significance level. 

The significance default is 95% limit, however can be changed by passing a different value to the 'alpha' argument in the plot_t2_control_chart function.
* 95% limit is $\alpha$ = 0.05 , $(1 - limit)$
* e.g. $limit$ = 80%, $1 - .8 = 0.2$


In [ ]:
gdf.plot_t2_control_chart(final_df['T2'].values, n_samples=final_df.shape[0], p_features=final_df.shape[1]-3, alpha=0.05)

Visualise Hotellings T^2 with an interactive chart using Plotly

In [ ]:
gdf.plot_t2_plotly(final_df['T2'].values, n_samples=final_df.shape[0], p_features=final_df.shape[1]-3)

4. Interpretation and Visualization

Models with T2>UCL are your "geodiverse" outliers—those that represent the edges of the model space.